# Week 9: Gradient Boost

## Setup and Data Loading

Importing libraries and loading the diabetes dataset used throughout this capstone.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0


## Train/Test Split

Splitting the data 80/20, consistent with prior weeks, to keep results comparable.

In [2]:
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(56553, 21)
(14139, 21)


## Baseline Gradient Boost Model

Training a Gradient Boosting Classifier with default hyperparameters to establish a baseline AUC-ROC score.

In [3]:
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)

y_pred_proba = gb_model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"Baseline Gradient Boost AUC-ROC: {auc_score:.4f}")

Baseline Gradient Boost AUC-ROC: 0.8316


## Testing Different Learning Rates

Comparing AUC-ROC across a range of learning rates (0.01, 0.05, 0.1, 0.3) to see how this hyperparameter affects model performance, following the tradeoff described in the Week 9 lesson: smaller learning rates need more estimators but may generalize better, while larger learning rates train faster but may be less refined.

In [4]:
learning_rates = [0.01, 0.05, 0.1, 0.3]

for lr in learning_rates:
    model = GradientBoostingClassifier(learning_rate=lr, random_state=42)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f"Learning rate {lr}: AUC-ROC = {auc:.4f}")

Learning rate 0.01: AUC-ROC = 0.8095
Learning rate 0.05: AUC-ROC = 0.8297
Learning rate 0.1: AUC-ROC = 0.8316
Learning rate 0.3: AUC-ROC = 0.8322


## Testing Number of Estimators and Tree Depth

Testing whether increasing the number of estimators helps a small learning rate (0.01) catch up in performance, and comparing different tree depths to observe the underfitting/overfitting tradeoff described in the lesson.

In [5]:
model_more_trees = GradientBoostingClassifier(learning_rate=0.01, n_estimators=300, random_state=42)
model_more_trees.fit(X_train, y_train)
y_pred_proba = model_more_trees.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Learning rate 0.01, 300 estimators: AUC-ROC = {auc:.4f}")

depths = [2, 3, 5, 8]
for depth in depths:
    model = GradientBoostingClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f"Max depth {depth}: AUC-ROC = {auc:.4f}")

Learning rate 0.01, 300 estimators: AUC-ROC = 0.8259
Max depth 2: AUC-ROC = 0.8304
Max depth 3: AUC-ROC = 0.8316
Max depth 5: AUC-ROC = 0.8321
Max depth 8: AUC-ROC = 0.8270


## Testing Regularization

Testing subsample and minimum samples per leaf as regularization techniques, following the methods described in the Week 9 lesson, to see whether they help control the overfitting observed at higher tree depths.

In [6]:
model_regularized = GradientBoostingClassifier(
    max_depth=8,
    subsample=0.8,
    min_samples_leaf=20,
    random_state=42
)
model_regularized.fit(X_train, y_train)
y_pred_proba = model_regularized.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Depth 8 with regularization: AUC-ROC = {auc:.4f}")

Depth 8 with regularization: AUC-ROC = 0.8281


## Final Tuned Gradient Boost Model

Combining the best-performing settings found through testing (learning rate 0.3, max depth 5) into a single tuned model, and comparing the result against all other models run in this capstone (KNN, SVM, decision tree, random forest, logistic regression).

In [7]:
best_model = GradientBoostingClassifier(
    learning_rate=0.3,
    max_depth=5,
    random_state=42
)
best_model.fit(X_train, y_train)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
best_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Tuned Gradient Boost (LR=0.3, depth=5): AUC-ROC = {best_auc:.4f}")
print()
print("Comparison across all models:")
print(f"Tuned Gradient Boost:    {best_auc:.4f}")
print(f"Baseline Gradient Boost: 0.8316")
print(f"Logistic Regression:     0.8232")
print(f"Random Forest:           0.8224")
print(f"KNN:                     0.8198")
print(f"SVM Linear:              0.8149")
print(f"Decision Tree:           0.8135")

Tuned Gradient Boost (LR=0.3, depth=5): AUC-ROC = 0.8278

Comparison across all models:
Tuned Gradient Boost:    0.8278
Baseline Gradient Boost: 0.8316
Logistic Regression:     0.8232
Random Forest:           0.8224
KNN:                     0.8198
SVM Linear:              0.8149
Decision Tree:           0.8135


## Revised Final Model

Testing learning rate and tree depth together (LR=0.3, depth=5) actually underperformed compared to either hyperparameter tuned individually. This suggests an interaction effect: pushing both settings toward more model complexity at once increases overfitting risk in a way neither setting caused alone. The best-performing single configuration found was learning rate 0.3 with default tree depth (3), AUC-ROC = 0.8322.

In [8]:
final_model = GradientBoostingClassifier(learning_rate=0.3, random_state=42)
final_model.fit(X_train, y_train)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Final Model (LR=0.3, default depth=3): AUC-ROC = {final_auc:.4f}")

Final Model (LR=0.3, default depth=3): AUC-ROC = 0.8322
